# NestedSimPy — M/M/1 demo

[NestedSimPy](https://nestedsimpy.github.io/) extends SimPy with *nested
simulation*. This notebook runs a small M/M/1 queue that forks **inner
simulations** at every arrival and reports the per-customer waiting-time
distribution.


## 1. Install

_NestedSimPy is released on a public GitHub repo; this installs it directly._


In [ ]:
!pip install -q "git+https://github.com/NestedSimPy/nestedsimpy.git"
import nestedsimpy
print("NestedSimPy ready —", len(nestedsimpy.__all__), "public objects")

## 2. Build and run a nested M/M/1


In [ ]:
import nestedsimpy
from nestedsimpy.reporting import wait_time_hook

ARRIVAL_RATE, SERVICE_RATE, SIM_TIME, SEED = 3.0, 4.0, 10.0, 42

def customer(env, server):
    with server.request() as req:
        yield req
        yield env.nested_timeout({"distribution": "exponential", "lambda": SERVICE_RATE})

def arrivals(env, server):
    while True:
        yield env.nested_timeout({"distribution": "exponential", "lambda": ARRIVAL_RATE})
        env.process(customer(env, server))

env = nestedsimpy.NestedEnvironment()
server = nestedsimpy.NestedResource(env, capacity=1, nested_id="srv")
env.process(arrivals(env, server))
env.set_output_options(out_dir="mm1_out", gzip_trace=False)
env.set_outer_seed(SEED)
env.set_nested_triggering_objects(nested_id="srv")
env.set_nesting_conditions({"on": "arrival", "frequency": 1})
env.set_inner_repetitions(2)
env.set_inner_stopping_condition(relative_time=5.0, triggering_customer_departs=True)
env.set_outer_stopping_condition(timeout=SIM_TIME)
env.set_postprocessor(
    lambda outer_dir, exports_dir, manifest, **o: {"waits": wait_time_hook(outer_dir, exports_dir, manifest, **o)},
    output_name="user_metrics.json")
env.nested_run()

## 3. The dataset it produces


In [ ]:
import glob, pandas as pd
path = glob.glob("mm1_out/**/user_waits.csv", recursive=True)[0]
waits = pd.read_csv(path)
waits[["branch_customer","branch_count","arrival","start","wait_time"]].head(12)

## 4. Waiting-time distribution


In [ ]:
import matplotlib.pyplot as plt
plt.hist(waits["wait_time"], bins=20)
plt.xlabel("waiting time"); plt.ylabel("number of inner runs")
plt.title("Per-branch waiting times (nested M/M/1)"); plt.show()
print(waits.groupby("branch_customer")["wait_time"].mean().head(10).round(3))